# Lesson 93 · Experiments that never forget — MLOps basics: experiment tracking with W&B, model versioning and Docker packaging

**Goal**: reproducible packaging with Docker + experiment tracking with W&B.

> **Checklist form**: versioning / monitoring / rollback — so your experiments never "forget".

🔗 Aurora connection: the training scripts in `src/aurora/` → a Docker image → the W&B Dashboard — the last step from research code to reproducible engineering.

← **Previous lesson** · [L92 · The end-to-end pipeline](L92_pipeline.ipynb)

> Last lesson covered **the end-to-end pipeline**: microphone → ASR → LLM → text answer, the whole chain assembled.
> This lesson explores **MLOps basics**.

## Learning goals

After this lesson you will be able to:

1. **Explain the size savings of a multi-stage Dockerfile**: tell the builder stage from the runtime stage, and explain how the layer cache works.
2. **Write complete W&B experiment-tracking code**: record hyperparameters and metrics with `wandb.init(mode='disabled')` / `wandb.log()` / `wandb.finish()`.
3. **Configure GitHub Actions CI/CD**: understand what triggers, the matrix strategy and artifact uploads do, and adapt them to the project's actual `.github/workflows/ci.yml`.
4. **Implement an inference-latency monitoring function**: compute the p50/p95 percentiles correctly and compare them with an SLA threshold, avoiding off-by-one errors.

Research code that runs on your own machine is only the starting point; real reproducibility means anyone, on any machine, running the same image gets the same result.
Docker solves "environment drift", W&B solves "experiment amnesia" — put the two together and your hyperparameter tuning leaves a record you can look up, and every model version can be traced back.

In [ ]:
# pip install wandb docker
try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False
    print("⚠️ wandb is not installed; training monitoring will be skipped")

## 1. Dockerfile best practices: multi-stage builds

**The problem**: baking development dependencies (pytest, black, jupyter) into the image bloats it to 2 GB+, which makes deployment slow and widens the attack surface.

**The fix**: a multi-stage build — the first stage, `builder`, installs and compiles every dependency; the second stage, `runtime`, copies over from `builder` only the files needed at run time.

`.dockerignore` excludes `__pycache__/`, `*.egg-info`, `.git/`, `notebooks/` and so on, to keep the build context from getting too big.

Key rules:
- `COPY pyproject.toml .` comes before `COPY src/ src/`, to take advantage of the layer cache — when the dependencies haven't changed, `pip install` is skipped
- Run as a non-root user: `RUN useradd -m aurora && USER aurora`
- Pin the base image version: `FROM python:3.11.9-slim` rather than `python:latest`

In [ ]:
# Demo: print a minimal working Dockerfile (nothing is actually built)
dockerfile = '''
# ── Stage 1: builder ──────────────────────────────
FROM python:3.11.9-slim AS builder
WORKDIR /build
COPY pyproject.toml README.md ./
COPY src/ src/
RUN pip install --no-cache-dir --prefix=/install .[audio]

# ── Stage 2: runtime ──────────────────────────────
FROM python:3.11.9-slim
COPY --from=builder /install /usr/local
RUN useradd -m aurora
USER aurora
ENTRYPOINT ["python", "-m", "aurora.serve"]  # aurora.serve is a minimal entry point, see src/aurora/serve.py
'''
print(dockerfile)

# .dockerignore example
dockerignore = '''
__pycache__/
*.egg-info/
.git/
notebooks/
*.pyc
.env
'''
# ⚠️ The aurora.serve in ENTRYPOINT already has a minimal stub at src/aurora/serve.py
print('--- .dockerignore ---')
print(dockerignore)

## 2. W&B experiment tracking: `wandb.init()` / `wandb.log()` / `wandb.save()`

`wandb.init(project='aurora-kws', config=cfg)` starts a run; `config` holds all the hyperparameters (lr, batch_size, n_mels and so on).

In the training loop, call `wandb.log({"loss": loss, "acc": acc, "step": step})` at every step, and W&B draws the line charts automatically.

When training ends, call `wandb.save("model.pt")` to upload the weights to the run's artifacts, so every model is matched one-to-one with its hyperparameters.

The three core lines:
```python
wandb.init(project='aurora-kws', config={'lr': 1e-3, 'batch_size': 32})
wandb.log({'loss': 0.42, 'val_acc': 0.91})
wandb.save('model.pt')
```

Once a run is complete, `wandb.finish()` closes the connection (optional in Jupyter; in scripts it's best to call it explicitly).

In [ ]:
# Demo: simulate a training run locally with wandb's 'disabled' mode (no network needed)
import math, random

cfg = {'lr': 1e-3, 'batch_size': 32, 'epochs': 5, 'n_mels': 64}

if HAS_WANDB:
    run = wandb.init(
        project='aurora-kws-demo',
        config=cfg,
        mode='disabled',   # offline demo, nothing is uploaded
        name='demo-run'
    )

# Simulated training loop
for epoch in range(cfg['epochs']):
    loss = 1.0 * math.exp(-0.6 * epoch) + random.uniform(0, 0.05)
    val_acc = 1 - 0.5 * math.exp(-0.8 * epoch) + random.uniform(-0.02, 0.02)
    if HAS_WANDB:
        wandb.log({'epoch': epoch, 'train_loss': round(loss, 4), 'val_acc': round(val_acc, 4)})
    print(f'epoch={epoch}  loss={loss:.4f}  val_acc={val_acc:.4f}')

# Simulate saving the model
# wandb.save('model.pt')  # uncomment when online

if HAS_WANDB:
    wandb.finish()
print('✅ W&B run finished (disabled mode, local demo only)')

## 3. Why track experiments: comparing hyperparameters and finding the best configuration

A single experiment's result tells you nothing on its own; only **multiple comparable runs** can answer "how much better is lr=1e-3 than lr=1e-4?".

W&B's **parallel coordinates plot** maps each set of hyperparameters to one polyline; the parameter combinations whose lines pass through the lowest val_loss region are the candidates for the best.

W&B **Sweep** automates this: define a search space → the Sweep Controller hands out jobs → several agents run experiments in parallel → the Dashboard aggregates them live.

For the Aurora KWS CNN, the two most important axes are:
- `lr`: decides how fast and how stably training converges (search range `1e-4` to `1e-2`)
- `batch_size`: affects gradient noise and training time (candidates `16 / 32 / 64`)

In [ ]:
# Demo: print a W&B Sweep configuration (the YAML equivalent, shown as a dict)
sweep_config = {
    'method': 'grid',
    'metric': {'name': 'val_acc', 'goal': 'maximize'},
    'parameters': {
        'lr': {
            'values': [1e-4, 1e-3, 1e-2]
        },
        'batch_size': {
            'values': [16, 32, 64]
        }
    }
}

import json as _json
print('sweep_config =')
print(_json.dumps(sweep_config, indent=2))
print()
print('grid search:', len(sweep_config['parameters']['lr']['values'])
      * len(sweep_config['parameters']['batch_size']['values']), 'experiments in total')

# How to run the Sweep when online:
# sweep_id = wandb.sweep(sweep_config, project='aurora-kws')
# wandb.agent(sweep_id, function=train_fn, count=9)
print('✅ Sweep configuration printed (uncomment to run it when online)')

## 4. GitHub Actions CI/CD: run lint + test automatically on every push

W&B tracks the experiment metrics and Docker packages the runtime environment — but who makes sure a code change doesn't break what already works?
The answer is **CI (continuous integration)**: every `git push` automatically triggers `make lint test`, and a failure blocks the merge.

Aurora's CI configuration lives in `.github/workflows/ci.yml`:

```yaml
name: Aurora CI
on:
  push:
    branches: [main, "claude/**"]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - name: Install
        run: pip install -e ".[dev]"
      - name: Lint
        run: make lint
      - name: Test
        run: make test
      - name: Upload W&B summary
        if: github.ref == 'refs/heads/main'
        env: { WANDB_API_KEY: "${{ secrets.WANDB_API_KEY }}" }
        run: python scripts/log_ci_metrics.py
```

**Key points**:
- `on: push` + `pull_request` covers all the usual triggers
- `secrets.WANDB_API_KEY` is injected through GitHub Secrets, never hard-coded
- `log_ci_metrics.py` uploads the test pass rate and lint error count to W&B, so they can be plotted side by side with the hyperparameter experiments

**Monitoring and alerts**: once deployed to production, you need to monitor inference latency and error rate:

```python
# scripts/log_ci_metrics.py
import wandb, subprocess, json

result = subprocess.run(["pytest", "--json-report", "--json-report-file=report.json"],
                        capture_output=True)
with open("report.json") as f:
    report = json.load(f)

wandb.init(project="aurora-ci", job_type="ci")
wandb.log({
    "tests/total":   report["summary"]["total"],
    "tests/passed":  report["summary"]["passed"],
    "tests/failed":  report["summary"]["failed"],
    "pass_rate":     report["summary"]["passed"] / report["summary"]["total"],
})
wandb.finish()
```

In [ ]:
import math
import json, os, subprocess
from pathlib import Path

# ── Generating the GitHub Actions workflow file (for the .github/workflows/ directory) ─────────────
# ⚠️ Teaching example: the CI YAML below differs from the project's actual .github/workflows/ci.yml
# The actual CI: matrix python 3.10/3.11/3.12, triggered on every branch, uses ruff check + pytest --cov
# To keep them in sync, refer to .github/workflows/ci.yml

CI_YAML = """\
name: Aurora CI

on:
  push:
    branches: [main, "claude/**"]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11"]

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip

      - name: Install
        run: pip install -e ".[dev]"

      - name: Lint
        run: make lint

      - name: Test with coverage
        run: |
          pip install pytest-json-report
          pytest --json-report --json-report-file=report.json -v

      - name: Upload test report
        uses: actions/upload-artifact@v4
        if: always()
        with:
          name: test-report
          path: report.json

      - name: Log to W&B
        if: github.ref == 'refs/heads/main'
        env:
          WANDB_API_KEY: ${{ secrets.WANDB_API_KEY }}
        run: python scripts/log_ci_metrics.py
"""

CI_SCRIPT = """\
#!/usr/bin/env python3
\"\"\"scripts/log_ci_metrics.py — upload CI test results to W&B\"\"\"
import json, os, wandb

with open("report.json") as f:
    report = json.load(f)

summary = report["summary"]
run = wandb.init(
    project="aurora-ci",
    job_type="ci",
    config={"python": "3.11", "branch": os.getenv("GITHUB_REF_NAME", "unknown")},
)
wandb.log({
    "tests/total":  summary["total"],
    "tests/passed": summary["passed"],
    "tests/failed": summary.get("failed", 0),
    "pass_rate":    summary["passed"] / max(summary["total"], 1),
})
wandb.finish()
print(f"✅ W&B upload complete  pass_rate={summary['passed']}/{summary['total']}")
"""

# Print only (nothing is written, so the project structure stays untouched)
print("=== .github/workflows/ci.yml ===")
print(CI_YAML)
print("=== scripts/log_ci_metrics.py ===")
print(CI_SCRIPT)

# ── Monitoring threshold check (can run directly in a local CI script) ──────────────────────────────────
def check_test_quality(report_path: str = "report.json",
                       min_pass_rate: float = 0.95) -> bool:
    """Read the pytest JSON report and check that the pass rate >= the threshold."""
    if not os.path.exists(report_path):
        print(f"⚠  {report_path} does not exist; skipping the check (pytest hasn't been run locally)")
        return True

    with open(report_path) as f:
        report = json.load(f)
    summary = report["summary"]
    pass_rate = summary["passed"] / max(summary["total"], 1)
    ok = pass_rate >= min_pass_rate
    status = "✅" if ok else "❌"
    print(f"{status} pass rate {pass_rate:.1%}  ({summary['passed']}/{summary['total']})  "
          f"threshold={min_pass_rate:.0%}")
    return ok

passed = check_test_quality()

# ── Simple latency monitoring (production services use Prometheus + Grafana; this shows the logic) ────────────────
import time, statistics

def monitor_inference_latency(n_trials: int = 20,
                              sla_p95_ms: float = 3000.0) -> dict:
    """Simulate N inferences, compute the p50 / p95 latency, and compare with the SLA."""
    import random
    random.seed(42)
    latencies = [random.gauss(2200, 200) for _ in range(n_trials)]
    latencies.sort()
    p50 = statistics.median(latencies)
    # Correct index: ceil(0.95*N)-1, avoiding the off-by-one that returns the maximum (p100)
    p95 = latencies[int(math.ceil(0.95 * n_trials)) - 1]
    assert p95 < latencies[-1], "p95 should not equal the maximum; check the index"
    ok  = p95 <= sla_p95_ms
    return {"p50_ms": p50, "p95_ms": p95, "sla_ok": ok, "n": n_trials}

perf = monitor_inference_latency()
status = "✅" if perf["sla_ok"] else "❌ SLA breached"
print(f"\n{status}  inference latency  p50={perf['p50_ms']:.0f}ms  "
      f"p95={perf['p95_ms']:.0f}ms  SLA={3000}ms")

## Parameter experiment: `lr` × `batch_size` grid search

**Search space**:
- `lr` ∈ `{1e-4, 1e-3, 1e-2}` — 3 learning rates (lr)
- `batch_size` ∈ `{16, 32, 64}` — 3 batch sizes
9 experiments in total, covering the full grid.

**What to expect** (this is what usually happens in real training; the local simulation below only models a penalty for lr straying from 1e-3 and less noise for bigger batches — 1e-4 and 1e-2 are penalised equally, and convergence speed isn't modelled):
- `lr=1e-2` + `batch_size=16`: lots of gradient noise + a learning rate on the high side, so the loss oscillates and val_acc is usually the lowest
- `lr=1e-3` + `batch_size=32`: the sweet spot by experience, with the highest peak val_acc
- `lr=1e-4` + `batch_size=64`: slowest to converge; val_acc hasn't topped out within 5 epochs

In the W&B Dashboard's **parallel coordinates plot**, the lines passing through the high-`val_acc` region cluster around `lr≈1e-3`, confirming the rule of thumb visually.

In [ ]:
# Simulate the 9-run grid search locally and look at how val_acc is distributed
import math, random, itertools

lrs = [1e-4, 1e-3, 1e-2]
batch_sizes = [16, 32, 64]
EPOCHS = 5

results = []

for lr, bs in itertools.product(lrs, batch_sizes):
    random.seed(int(lr * 1e5) + bs)   # fixed seed for reproducibility
    noise_scale = 0.05 / (bs ** 0.5)  # bigger batch, less noise
    lr_penalty = abs(math.log10(lr) + 3) * 0.06  # the further from 1e-3, the bigger the penalty
    final_acc = 0.91 - lr_penalty + random.uniform(-noise_scale, noise_scale)
    final_acc = round(min(max(final_acc, 0.6), 0.99), 4)
    results.append((lr, bs, final_acc))

# Print in descending order of val_acc
results.sort(key=lambda x: -x[2])
print(f"{'lr':>8}  {'batch_size':>10}  {'val_acc':>8}")
print('-' * 34)
for lr, bs, acc in results:
    marker = ' ← best' if acc == results[0][2] else ''
    print(f"{lr:>8.1e}  {bs:>10}  {acc:>8.4f}{marker}")

print()
print('✅ Grid search simulation complete')
print(f'Best configuration: lr={results[0][0]:.1e}, batch_size={results[0][1]}, val_acc={results[0][2]}')

## 5. Model versioning: making every experiment reproducible

Saving a `.pt` file after every training run quickly gets out of hand.
Model versioning binds together the weight file, the hyperparameters, the dataset version and the code commit.

### The simplest practice: W&B Artifacts

```python
artifact = wandb.Artifact("kws-model", type="model",
                           metadata={"val_acc": 0.913, "config": config})
artifact.add_file("checkpoints/best.pt")
run.log_artifact(artifact)
# to restore later: wandb artifact get aurora/kws-model:v3
```

### Splitting the work between Git and Artifacts

| Goes into Git | Stays out of Git (goes into Artifacts / DVC) |
|---|---|
| code, configs, Dockerfile | model weights (> 50 MB) |
| `pyproject.toml` | raw datasets |
| experiment config YAML | W&B run logs |

In [ ]:
# Print a standard .gitignore (for Aurora)
lines = [
    "# Model weights", "checkpoints/", "*.pt", "*.onnx", "*.bin", "",
    "# Datasets", "data/raw/", "data/cache/", "*.wav", "*.flac", "",
    "# W&B", "wandb/", "",
    "# Python", "__pycache__/", "*.pyc", ".venv/", "dist/", "*.egg-info/", "",
    "# Jupyter", ".ipynb_checkpoints/", "",
    "# System", ".DS_Store",
]
print("\n".join(lines))

## 6. ✏️ Exercise: write a complete tracking config for one KWS training run

In [ ]:
# ✏️ Fill in your experiment config (so this experiment can be reproduced whenever it's run)
experiment_config = {
    "project":       "aurora-kws",
    "run_name":      None,   # e.g. "mel80-lr1e3-bs32-e20"
    "dataset":       None,   # e.g. "google_speech_commands_v2"
    "n_mels":        None,
    "win_len_ms":    None,
    "hop_ms":        None,
    "lr":            None,
    "batch_size":    None,
    "epochs":        None,
    "optimizer":     None,
    "best_val_acc":  None,
    "test_acc":      None,
}
filled = sum(1 for v in experiment_config.values() if v is not None)
print(f"Config completeness: {filled}/{len(experiment_config)}")
if experiment_config["run_name"]:
    print(f"Run: {experiment_config['run_name']}")

In [ ]:
# ✏️ Advanced exercise: implement the setup_experiment() function
# Requirements: take an experiment config dict, initialize a W&B run (disabled mode), return the run object
# Check: run.config['n_mels'] == cfg['n_mels']

def setup_experiment(cfg: dict):
    """
    Initialize one W&B experiment run.
    Parameters:
        cfg: hyperparameter config dict, with at least the keys 'project', 'run_name', 'n_mels'
    Returns:
        a wandb.Run object (disabled mode, no network needed)
    """
    # TODO: guard with HAS_WANDB, and call wandb.init() in disabled mode
    # TODO: pass project=cfg['project'], name=cfg['run_name'], config=cfg, mode='disabled'
    # TODO: return the run object
    raise NotImplementedError('Please implement setup_experiment()')

# ── Check (don't modify the code below) ──────────────────────────────────────────
test_cfg = {
    'project': 'aurora-kws',
    'run_name': 'mel64-lr1e3-bs32-e10',
    'n_mels': 64,
    'lr': 1e-3,
    'batch_size': 32,
    'epochs': 10,
}
if HAS_WANDB:
    try:
        run = setup_experiment(test_cfg)
        assert run is not None, 'setup_experiment() should not return None'
        assert run.config['n_mels'] == test_cfg['n_mels'], \
            f"n_mels mismatch: expected {test_cfg['n_mels']}, got {run.config['n_mels']}"
        assert run.config['lr'] == test_cfg['lr'], \
            f"lr mismatch: expected {test_cfg['lr']}, got {run.config['lr']}"
        run.finish()
        print('✅ setup_experiment() check passed')
    except (NotImplementedError, TypeError):
        print('⏳ Implement setup_experiment(), then run this again')
    except AssertionError as e:
        print(f'❌ Check failed: {e}')
else:
    print('⚠️ wandb is not installed; skipping the setup_experiment() check')

## Wrap-up

This lesson strung experiment records, image packaging and versioning into a single production chain: `W&B` keeps experiments from forgetting, `Docker` keeps environments from drifting.
Only when training, metrics and deployment can all be traced back does Aurora's research code truly reach reproducible engineering.
Next lesson: **L94** Aurora v1 demo — turning six months of work into a story you can explain clearly, show off, and back up with evidence.

---

→ **Next lesson** · [L94 · Aurora v1 demo](L94_demo.ipynb)

> Next lesson covers **the Aurora v1 demo**: every capability in one showcase, and organizing what you've built into work and ideas you can explain clearly.